In [ ]:
!pip uninstall -y transformers datasets
!pip install transformers==4.56.1 datasets

Found existing installation: transformers 4.56.1
Uninstalling transformers-4.56.1:
  Successfully uninstalled transformers-4.56.1
Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
  Using cached transformers-4.56.1-py3-none-any.whl.metadata (42 kB)
  Using cached datasets-4.0.0-py3-none-any.whl.metadata (19 kB)
Using cached transformers-4.56.1-py3-none-any.whl (11.6 MB)
Using cached datasets-4.0.0-py3-none-any.whl (494 kB)


In [ ]:
import transformers, datasets
print(transformers.__version__)
print(datasets.__version__)

4.56.1
4.0.0


# Colab setup & Drive mount

In [ ]:
!nvidia-smi -L || echo "No GPU? Go to Runtime → Change runtime type → GPU"
!python -V
!pip -q install "transformers>=4.44.0" datasets accelerate sentencepiece tensorboard

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

BASE="/content/drive/MyDrive/data_colab/miniP"
print("Base folder:", BASE)

GPU 0: Tesla T4 (UUID: GPU-a5fbcef6-4b90-29b5-a2aa-6e17a8c39fc5)
Python 3.12.11
Mounted at /content/drive
Base folder: /content/drive/MyDrive/data_colab/miniP


In [ ]:
# 1️⃣ Imports
import os
from datasets import load_from_disk
from transformers import (
    AutoConfig, AutoModelForCausalLM,
    Trainer, TrainingArguments,
    DataCollatorForLanguageModeling
)
from transformers import default_data_collator

# Training dataset according to HF format (.arrow)

In [ ]:
# # Streaming dataset directly
# # ⚠️ But note: you lose random access & .select(range(5000)) won’t work.
# # 👉 Downside: with streaming=True, you can’t .map() easily, shuffling is limited, and Trainer expects non-streaming datasets.

# from datasets import load_dataset

# files = {
#     "train": f"{BASE}/data/multilang_train.tok",
#     "validation": f"{BASE}/data/multilang_val.tok",
#     "test": f"{BASE}/data/multilang_test.tok",
# }

# ds = load_dataset("text", data_files=files, streaming=True)

In [ ]:
# -----------------------------
# 2️⃣ Load pre-tokenized dataset
# -----------------------------
from datasets import load_from_disk
ds = load_from_disk(f"{BASE}/data/multilang_tok_dataset")

Loading dataset from disk:   0%|          | 0/98 [00:00<?, ?it/s]

In [ ]:
print(ds["train"][0])

{'input_ids': [468, 102, 51, 733, 6421, 109, 200, 39868, 5286, 4], 'labels': [468, 102, 51, 733, 6421, 109, 200, 39868, 5286, 4]}


# Pick a Gemma 3 architecture and shrink to ~125M params

In [ ]:
from google.colab import userdata
import os

hf_token = userdata.get("HF_TOKEN")
print("Token loaded:", hf_token[:10] + "..." if hf_token else "❌ Not found")

# Set environment variable for Hugging Face
if hf_token:
    os.environ["HF_TOKEN"] = hf_token

Token loaded: hf_LIaCdEi...


In [ ]:
# 3️⃣ Define Gemma3 model from scratch
from transformers import AutoConfig, AutoModelForCausalLM

cfg = AutoConfig.from_pretrained("google/gemma-3-1b-pt", use_auth_token=os.environ["HF_TOKEN"])  # Gemma 3 (dense) implementation

# Customize for your 134M model
cfg.vocab_size = 50_000
cfg.hidden_size = 640
cfg.num_hidden_layers = 12
cfg.num_attention_heads = 10
cfg.intermediate_size = 2560
cfg.max_position_embeddings = 1024  # context length you’ll train with
cfg.tie_word_embeddings = True      # tie LM head to embeddings (saves params)

# Build the model from scratch (no pretrained weights)
model = AutoModelForCausalLM.from_config(cfg, attn_implementation="eager")

# Enable gradient checkpointing to save memory
model.gradient_checkpointing_enable()
total_params = sum(p.numel() for p in model.parameters())
print(f"Model built. Total params: {total_params/1e6:.2f}M")

/usr/local/lib/python3.12/dist-packages/transformers/models/auto/configuration_auto.py:1273: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


Model built. Total params: 134.27M


# Training

In [ ]:
# 4️⃣ Data collator
# data_collator = DataCollatorForLanguageModeling(
#     tokenizer=None,  # dataset already tokenized
#     mlm=False        # causal LM
# )
data_collator = default_data_collator  # simply collates tensors as-is

In [ ]:
# 5️⃣ Training arguments
training_args = TrainingArguments(
    output_dir=f"{BASE}/gemma3_checkpoints",
    per_device_train_batch_size=1,        # small batch due to Colab memory
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,       # simulate larger batch
    eval_strategy="steps",
    eval_steps=200,
    save_steps=500,
    save_total_limit=3,                   # keep only last 3 checkpoints
    logging_steps=50,
    num_train_epochs=1,                   # increase later
    fp16=False,                           # mixed precision
    report_to="none",                     # avoid wandb if not needed
    push_to_hub=False,                    # set True if you want to push later
    max_steps=2000,                       # train only for N steps instead of full epoch
)

# 6️⃣ Trainer
trainer = Trainer(
    model=model,
    # train_dataset=ds["train"],
    # eval_dataset=ds["validation"],
    train_dataset=ds["train"].select(range(5000)),
    eval_dataset=ds["validation"].select(range(1000)),
    data_collator=data_collator,
    tokenizer=None,
    args=training_args
)

/tmp/ipython-input-4085020211.py:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# 7️⃣ Start training (resume if checkpoint exists)
import glob
checkpoints = glob.glob(f"{BASE}/gemma3_checkpoints/checkpoint-*")
last_ckpt = max(checkpoints, default=None, key=os.path.getctime)

if last_ckpt:
    print(f"Resuming from checkpoint: {last_ckpt}")
    trainer.train(resume_from_checkpoint=last_ckpt)
else:
    print("No checkpoint found, starting fresh training.")
    trainer.train()

# ✅ Save final model
trainer.save_model(f"{BASE}/gemma3_final")
print("Training completed. Model saved!")

No checkpoint found, starting fresh training.


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
200,7.676700,nan
400,6.965400,nan
600,6.884000,nan
800,6.319700,nan
1000,5.779900,nan
1200,5.806900,nan
1400,5.396000,nan
1600,5.275700,nan
1800,5.224100,nan
2000,5.178900,nan


Training completed. Model saved!


In [ ]:
metrics = trainer.evaluate(ds["validation"])
print(metrics)

In [ ]:
tok.save_pretrained(f"{BASE}/my_tokenizer")

('./my_tokenizer/tokenizer_config.json',
 './my_tokenizer/special_tokens_map.json',
 './my_tokenizer/tokenizer.model',
 './my_tokenizer/added_tokens.json')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# ✅ load tokenizer from local folder
tok = AutoTokenizer.from_pretrained(f"{BASE}/my_tokenizer", use_fast=True)

# ✅ load trained model
trained_model = AutoModelForCausalLM.from_pretrained(f"{BASE}/gemma3_final")

# build generator pipeline
generator = pipeline("text-generation", model=trained_model, tokenizer=tok)

# try generating text
print(generator("Hello, my name is", max_new_tokens=50, do_sample=True, top_k=50, top_p=0.95))

Device set to use cuda:0


[{'generated_text': "Hello, my name is.... in of of the the their,. in in the, the the to the- the the'ss.. as, the would and of the a to the this. and the, as. the’,"}]


In [ ]:
from transformers import LlamaTokenizer, AutoModelForCausalLM, pipeline

# 🔹 Load tokenizer from your SentencePiece model
TOK_DIR = "drive/MyDrive/data_colab/miniP"

tok = LlamaTokenizer(
    vocab_file=f"{TOK_DIR}/spm_multilang.model",
    bos_token="<s>",
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>",
    legacy=False
)

# 🔹 Load your fine-tuned model
trained_model = AutoModelForCausalLM.from_pretrained("./gemma3_final")

# 🔹 Build pipeline
generator = pipeline("text-generation", model=trained_model, tokenizer=tok)

# 🔹 Test generation
print(generator("Hello, my name is", max_length=10, do_sample=True))

Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Both `max_new_tokens` (=256) and `max_length`(=10) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Hello, my name is., thissss to the,s and in and a to it and..... their is and.. with. of the, as a and a to its of the and,. made. to the to the a a a as.. as it the the the the a a and to to the a to of the a for the of thes of the the the a..... the a.. of the.. of the the the as. do. to under the into to ins and and and the to the a. of of of. has........ of the to......th the the a a a a a in to the the the to that the, in that’ of of of of of of of of of, on of...... the the with, as of the,,, when the the to a a.. the a to to the... to became And.. a of of the of the,, the. to the in of of the,,, a.. a a of the. the the of the of of a be of of of the a....'}]


# Main training on full dataset

In [ ]:
!pip uninstall -y transformers datasets
!pip install transformers==4.56.1 datasets

In [3]:
import transformers, datasets
print(transformers.__version__)
print(datasets.__version__)

4.56.1
4.0.0


In [5]:
!nvidia-smi -L || echo "No GPU? Go to Runtime → Change runtime type → GPU"
!python -V
!pip -q install "transformers>=4.44.0" datasets accelerate sentencepiece tensorboard

from google.colab import drive
# drive.mount('/content/drive', force_remount=True)
drive.mount('/content/drive')

BASE="/content/drive/MyDrive/data_colab/miniP"
print("Base folder:", BASE)

GPU 0: Tesla T4 (UUID: GPU-84de3079-c6b8-4bbd-13fe-d88d48a57152)
Python 3.12.11
Base folder: /content/drive/MyDrive/data_colab/miniP


In [ ]:
# 1️⃣ Imports
import os
from datasets import load_from_disk
from transformers import (
    AutoConfig, AutoModelForCausalLM,
    Trainer, TrainingArguments,
    DataCollatorForLanguageModeling
)
from transformers import default_data_collator
from torch.nn.utils.rnn import pad_sequence
import torch

In [ ]:
# 2️⃣ Load pre-tokenized dataset
from datasets import load_from_disk
ds = load_from_disk(f"{BASE}/data/multilang_tok_dataset")

print(ds["train"][0])

Loading dataset from disk:   0%|          | 0/98 [00:00<?, ?it/s]

{'input_ids': [468, 102, 51, 733, 6421, 109, 200, 39868, 5286, 4], 'labels': [468, 102, 51, 733, 6421, 109, 200, 39868, 5286, 4]}


In [ ]:
from transformers import AutoConfig, AutoModelForCausalLM, Trainer, TrainingArguments
import os, glob, torch
from torch.nn.utils.rnn import pad_sequence

# Load HF token
from google.colab import userdata
hf_token = userdata.get("HF_TOKEN")
if hf_token: os.environ["HF_TOKEN"] = hf_token
print("Token loaded:", hf_token[:10] + "..." if hf_token else "❌ Not found")

# Define Gemma3 config (~134M)
cfg = AutoConfig.from_pretrained("google/gemma-3-1b-pt", token=os.environ["HF_TOKEN"])
cfg.vocab_size = 50_000
cfg.hidden_size = 640
cfg.num_hidden_layers = 12
cfg.num_attention_heads = 10
cfg.intermediate_size = 2560
cfg.max_position_embeddings = 1024
cfg.tie_word_embeddings = True

model = AutoModelForCausalLM.from_config(cfg, attn_implementation="eager")
print(f"Model built. Total params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

# Collator (pad safely)
def collate_fn(batch):
    input_ids = [torch.tensor(x["input_ids"], dtype=torch.long) for x in batch]
    labels    = [torch.tensor(x["labels"], dtype=torch.long) for x in batch]
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=0)
    labels    = pad_sequence(labels, batch_first=True, padding_value=-100)
    return {"input_ids": input_ids, "labels": labels}

# TrainingArguments
training_args = TrainingArguments(
    output_dir=f"{BASE}/gemma3_checkpoints",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    dataloader_num_workers=0,    # avoids multiprocessing issues on Colab/Mac

    # Use the older arg name expected by your Transformers: eval_strategy
    eval_strategy="no",          # disable validation to speed up
    save_strategy="steps",
    save_steps=5000,            # checkpoint frequency (tune for dataset size)
    save_total_limit=2,
    save_only_model=False,

    logging_steps=1000,
    num_train_epochs=1,          # full pass over the dataset
    max_steps=-1,                # no artificial step cap (train full dataset)

    fp16=False,
    report_to="none",
    push_to_hub=False,
)

# Trainer
trainer = Trainer(
    model=model,
    train_dataset=ds["train"],  # your dataset here
    data_collator=collate_fn,
    tokenizer=None,
    args=training_args,
)

# Resume if checkpoint exists
checkpoints = glob.glob(f"{BASE}/gemma3_checkpoints/checkpoint-*")
last_ckpt = max(checkpoints, default=None, key=os.path.getctime)

if last_ckpt:
    print(f"▶️ Resuming from checkpoint: {last_ckpt}")
    trainer.train(resume_from_checkpoint=last_ckpt)
else:
    print("🚀 Starting fresh training.")
    trainer.train()

# Save final model
trainer.save_model(f"{BASE}/gemma3_final")
print("✅ Training completed. Model saved!")

Token loaded: hf_LIaCdEi...


config.json:   0%|          | 0.00/880 [00:00<?, ?B/s]

Model built. Total params: 134.27M
▶️ Resuming from checkpoint: /content/drive/MyDrive/data_colab/miniP/gemma3_checkpoints/checkpoint-60000


/tmp/ipython-input-622656849.py:66: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
There were missing keys in the checkpoint model loaded: ['lm_head.weight'].
	save_steps: 5000 (from args) != 20000 (from trainer_state.json)


Step,Training Loss
61000,6.661700
62000,6.615400
63000,6.691400
64000,6.653100
65000,6.588500
66000,6.562800
67000,6.620500
68000,6.579000
69000,6.613200
70000,6.701500


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.35 GiB. GPU 0 has a total capacity of 14.74 GiB of which 794.12 MiB is free. Process 2333 has 13.96 GiB memory in use. Of the allocated memory 13.21 GiB is allocated by PyTorch, and 631.70 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [7]:
from transformers import LlamaTokenizer, AutoModelForCausalLM, pipeline

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

# 🔹 Load tokenizer from your SentencePiece model
TOK_DIR = "drive/MyDrive/data_colab/miniP"

tok = LlamaTokenizer(
    vocab_file=f"{TOK_DIR}/spm_multilang.model",
    bos_token="<s>",
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>",
    legacy=False
)

gen = pipeline("text-generation", model=f"{BASE}/gemma3_checkpoints/checkpoint-60000", tokenizer=tok, device=0 if device=="cuda" else -1)
print(gen("Hello, how are you?", max_new_tokens=50))
# print(gen("Hello, how are", max_new_tokens=2))
print(gen("नमस्ते, आप कैसे हैं?", max_new_tokens=50))
# print(gen("नमस्ते, आप कैसे", max_new_tokens=2))
print(gen("হ্যালো, তুমি কেমন আছো?", max_new_tokens=50))
# print(gen("হ্যালো, তুমি কেমন", max_new_tokens=2))

Device set to use cuda:0


[{'generated_text': 'Hello, how are you? a good. in the two.- compress-up-based-day-based-based-based-based- seasonal-I-up-based-up-like-to-based-based-i-year-sin, and'}]
[{'generated_text': 'नमस्ते, आप कैसे हैं? से बाहर खूब भी है।।, जो एक ही एक इस दौरान में इस दौरान भी को देखा हैं.।।।। भी भी की घर के साथ वह यह है, तो इस ही यह यह एक अलग भी भी किसी वीडियो है। वह'}]
[{'generated_text': 'হ্যালো, তুমি কেমন আছো? কাজে, ২০২১,: পুঁজি:৪ের জন্য করছেনক পারেন: ভাই হয়কা পারেন, তিনি বিভিন্ন, নি, তা,  করছেনর তাকে, জানা নারী, -টিঃ, ছিল করে, তিনি নিয়ে নিয়ে নিয়ে নিয়ে নিয়ে'}]
